In [10]:
import os
import glob
import datetime
import numpy as np
from netCDF4 import Dataset
from scipy import stats
import matplotlib.pyplot as plt

def calculate_spi(rainfall_data, scale_months=3):
    """SPI 계산 함수
    rainfall_data: 3차원 배열 (time, latitude, longitude)
    scale_months: 이동평균에 사용할 시간 스케일 (예, 3 => SPI-3)
    """
    def fit_gamma_distribution(data):
        data = np.array(data)
        data = data[data > 0]  # 0 값 제외
        if len(data) < 2:
            return None, None
        # 감마분포 피팅 (loc=0 고정)
        shape, loc, scale_val = stats.gamma.fit(data, floc=0)
        return shape, scale_val

    def calculate_spi_values(data, shape, scale_val):
        if shape is None or scale_val is None:
            return np.nan
        # 누적분포함수 계산 후 정규분포의 역함수를 통해 SPI 산출
        cdf = stats.gamma.cdf(data, shape, loc=0, scale=scale_val)
        spi = stats.norm.ppf(cdf)
        return spi

    rainfall_array = np.array(rainfall_data)
    spi_values = np.zeros_like(rainfall_array)  # 원본과 동일한 shape로 초기화

    # 각 격자점에 대해 SPI 계산 (이동평균으로 인해 유효한 값은 index scale_months-1부터 채워짐)
    for i in range(rainfall_array.shape[1]):
        for j in range(rainfall_array.shape[2]):
            timeseries = rainfall_array[:, i, j]
            # 이동평균 계산: mode='valid'로 인해 길이는 (N - scale_months + 1)
            rolling_mean = np.convolve(timeseries, np.ones(scale_months) / scale_months, mode='valid')
            shape, scale_val = fit_gamma_distribution(rolling_mean)
            if shape is not None:
                spi = calculate_spi_values(rolling_mean, shape, scale_val)
                # 결과를 원본 배열의 유효 시간부분에 저장 (앞의 scale_months-1 자리는 0 혹은 nan)
                spi_values[scale_months - 1:, i, j] = spi
            else:
                spi_values[scale_months - 1:, i, j] = np.nan

    return spi_values

def save_individual_spi_to_nc(spi_values, dates, output_folder, scale_months):
    """
    각 시간별(월별) SPI 값을 개별 NetCDF 파일로 저장하는 함수.
    - spi_values: 유효한 SPI 값 배열 (time, latitude, longitude)
    - dates: 해당 시간의 datetime 리스트 (length가 spi_values의 time dimension과 일치)
    - output_folder: 저장할 폴더 (존재하지 않으면 생성)
    - scale_months: SPI 스케일 (파일 metadata에 기록)
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        
    # 예시로 제주 지역의 위도/경도 범위를 사용 (필요시 수정)
    jeju_lat_start, jeju_lat_end = 33.1, 33.6
    jeju_lon_start, jeju_lon_end = 126.1, 127.1
    lat_vals = np.linspace(jeju_lat_start, jeju_lat_end, spi_values.shape[1])
    lon_vals = np.linspace(jeju_lon_start, jeju_lon_end, spi_values.shape[2])
    
    # 각 시간별로 파일 저장 (예: "SPI_3month_201603.nc")
    for idx, date in enumerate(dates):
        filename = os.path.join(output_folder, f"SPI_{scale_months}month_{date.strftime('%Y%m')}.nc")
        with Dataset(filename, 'w', format='NETCDF4') as nc:
            # 단일 시간만 저장하므로 시간 차원 길이는 1
            nc.createDimension('time', 1)
            nc.createDimension('latitude', len(lat_vals))
            nc.createDimension('longitude', len(lon_vals))
            
            times = nc.createVariable('time', 'i4', ('time',))
            lats = nc.createVariable('latitude', 'f4', ('latitude',))
            lons = nc.createVariable('longitude', 'f4', ('longitude',))
            spi_var = nc.createVariable('spi', 'f4', ('time', 'latitude', 'longitude',))
            
            times[:] = [0]  # 단일 시간
            lats[:] = lat_vals
            lons[:] = lon_vals
            # 단일 시간에 해당하는 SPI 공간 데이터: shape (1, lat, lon)
            spi_var[:] = spi_values[idx:idx+1, :, :]
            
            nc.description = f'SPI-{scale_months} values for {date.strftime("%Y-%m")}'
            nc.created = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            times.units = f'days since {date.strftime("%Y-%m-%d")}'
        
        print(f"Saved {filename}")

def load_rainfall_data(nc_file, variable_name='precipitation'):
    """
    단일 NetCDF 파일에서 월간 강수량 자료를 로딩하는 함수.
    파일 내 변수 이름은 기본적으로 'monthly_sum'으로 가정합니다.
    실제 변수명이 다르다면 variable_name 인자를 수정하세요.
    """
    with Dataset(nc_file, 'r') as nc:
        available_vars = list(nc.variables.keys())
        if variable_name not in available_vars:
            raise KeyError(f"Variable '{variable_name}' not found. Available variables: {available_vars}")
        rainfall = nc.variables[variable_name][:]
    return rainfall

def parse_date_from_filename(nc_file):
    """
    파일명 'yyyymm_monthly_sum.nc'에서 날짜 정보를 파싱하여 datetime 객체로 반환
    """
    base = os.path.basename(nc_file)
    date_str = base[:6]  # 처음 6자리: yyyymm
    year = int(date_str[:4])
    month = int(date_str[4:6])
    return datetime.datetime(year, month, 1)

def main():
    # 입력 폴더 및 출력 폴더 경로 설정
    input_folder = r'G:\24.11.Graduate\1.CoKriging(Monthly)'
    output_folder = r'C:\25.01.Drought\SPI(Co-Kriging)'
    scale_months = 3  # SPI-3 (3개월 누적)
    
    file_pattern = os.path.join(input_folder, '*_monthly_sum.nc')
    files = sorted(glob.glob(file_pattern))
    
    rainfall_data_list = []
    dates = []
    
    # 파일명으로부터 날짜를 파싱하고 지정 기간 내의 파일만 사용 (2016-01부터 2023-12)
    for nc_file in files:
        date = parse_date_from_filename(nc_file)
        if datetime.datetime(2016, 1, 1) <= date <= datetime.datetime(2023, 12, 31):
            # 만약 파일 내부 변수명이 'monthly_sum'이 아니라면 이 인자를 변경하세요.
            data = load_rainfall_data(nc_file, variable_name='precipitation')
            # 데이터가 2차원(위도, 경도)라면 time 차원을 추가합니다.
            if data.ndim == 2:
                data = data[np.newaxis, ...]
            rainfall_data_list.append(data)
            dates.append(date)
    
    # 입력 파일이 월별 데이터이므로, 시간축으로 결합 시 shape는 (총 월수, lat, lon)
    rainfall_array = np.concatenate(rainfall_data_list, axis=0)
    print(f"Total time steps: {rainfall_array.shape[0]}")
    
    # SPI 계산
    spi_values_all = calculate_spi(rainfall_array, scale_months=scale_months)
    # 계산 결과는 rolling mean 때문에 앞의 (scale_months-1) time steps는 유효하지 않음
    spi_values_valid = spi_values_all[scale_months - 1:, :, :]  # 예: 96 - 2 = 94 time steps
    spi_dates_valid = dates[scale_months - 1:]
    
    # 각 유효 시간별 결과를 개별 파일로 저장 (총 94개 파일 생성됨)
    save_individual_spi_to_nc(spi_values_valid, spi_dates_valid, output_folder, scale_months)

if __name__ == "__main__":
    main()


Total time steps: 91
Saved C:\25.01.Drought\SPI(Co-Kriging)\SPI_3month_201603.nc
Saved C:\25.01.Drought\SPI(Co-Kriging)\SPI_3month_201604.nc
Saved C:\25.01.Drought\SPI(Co-Kriging)\SPI_3month_201605.nc
Saved C:\25.01.Drought\SPI(Co-Kriging)\SPI_3month_201606.nc
Saved C:\25.01.Drought\SPI(Co-Kriging)\SPI_3month_201607.nc
Saved C:\25.01.Drought\SPI(Co-Kriging)\SPI_3month_201608.nc
Saved C:\25.01.Drought\SPI(Co-Kriging)\SPI_3month_201609.nc
Saved C:\25.01.Drought\SPI(Co-Kriging)\SPI_3month_201610.nc
Saved C:\25.01.Drought\SPI(Co-Kriging)\SPI_3month_201611.nc
Saved C:\25.01.Drought\SPI(Co-Kriging)\SPI_3month_201612.nc
Saved C:\25.01.Drought\SPI(Co-Kriging)\SPI_3month_201701.nc
Saved C:\25.01.Drought\SPI(Co-Kriging)\SPI_3month_201702.nc
Saved C:\25.01.Drought\SPI(Co-Kriging)\SPI_3month_201703.nc
Saved C:\25.01.Drought\SPI(Co-Kriging)\SPI_3month_201704.nc
Saved C:\25.01.Drought\SPI(Co-Kriging)\SPI_3month_201705.nc
Saved C:\25.01.Drought\SPI(Co-Kriging)\SPI_3month_201706.nc
Saved C:\25.01.Drou